In [9]:
import duckdb
import plotly.express as px
import os

# PERCORSO VERIFICATO (Dalla tua ricerca automatica)
parquet_path = '../../../out/data/clean/istat-delitti-denunciati/2024/istat-delitti-denunciati_2024_clean.parquet'

# Verifica rapida
if os.path.exists(parquet_path):
    print("✅ FILE TROVATO! Procedo con l'analisi...")
    con = duckdb.connect()
    
    # 1. Grafico dei reati più comuni (Usiamo il 2024 perché c'è solo quello nel file)
    df_reati = con.execute(f"""
        SELECT codice_reato, SUM(CAST(numero_denunce AS DOUBLE)) as totale
        FROM '{parquet_path}'
        WHERE anno = '2024'
        GROUP BY codice_reato
        ORDER BY totale DESC
        LIMIT 15
    """).df()

    fig1 = px.bar(df_reati, x='codice_reato', y='totale', 
                 title='Top 15 Reati - Anno 2024',
                 template='plotly_white')
    fig1.show()

    # 2. Focus su Rapine (ROBBER) - Sostituiamo ARSON con un codice che esiste
    df_trend = con.execute(f"""
        SELECT anno, SUM(CAST(numero_denunce AS DOUBLE)) as totale
        FROM '{parquet_path}'
        WHERE codice_reato = 'ROBBER'
        GROUP BY anno
        ORDER BY anno
    """).df()

    fig2 = px.line(df_trend, x='anno', y='totale', 
                  title='Trend Rapine (ROBBER)',
                  markers=True)
    fig2.show()
else:
    print(f"❌ Errore critico: il file {parquet_path} non esiste.")

✅ FILE TROVATO! Procedo con l'analisi...


In [10]:
# Analisi comparativa dei trend richiesta (2010-2015)
codici_da_analizzare = ('ROBBER', 'MAFIAHOM', 'CARTHEF', 'RAPE')

df_all_trends = con.execute(f"""
    SELECT 
        CAST(anno AS INTEGER) as anno, 
        codice_reato, 
        SUM(CAST(numero_denunce AS DOUBLE)) as totale
    FROM '{parquet_path}'
    WHERE codice_reato IN {codici_da_analizzare} 
      AND anno BETWEEN '2010' AND '2015' -- Filtro corretto per la PR
    GROUP BY anno, codice_reato
    ORDER BY anno
""").df()

if not df_all_trends.empty:
    fig_trend = px.line(df_all_trends, x='anno', y='totale', color='codice_reato',
                       title='Evoluzione Comparata dei Reati (2010-2015)',
                       markers=True, template='plotly_white')
    fig_trend.show()

In [11]:
# Analisi territoriale per Incendi Boschivi (2024)
# Usiamo i codici che abbiamo trovato: '2024' e 'FOREARS'
df_geo = con.execute(f"""
    SELECT 
        codice_territorio, 
        SUM(CAST(numero_denunce AS DOUBLE)) as totale
    FROM '{parquet_path}'
    WHERE anno = '2024' 
      AND codice_reato = 'FOREARS' 
    GROUP BY codice_territorio
    ORDER BY totale DESC
    LIMIT 20
""").df()

if df_geo.empty:
    print("⚠️ Nessun dato trovato. Controlla che la pipeline abbia generato i dati correttamente.")
else:
    fig_geo = px.bar(df_geo, 
                    x='codice_territorio', 
                    y='totale', 
                    title='Top 20 Territori per Incendi Boschivi (2024)',
                    labels={'totale': 'Totale Denunce', 'codice_territorio': 'Provincia/Regione'},
                    color='totale', 
                    color_continuous_scale='Reds',
                    template='plotly_white')
    
    fig_geo.update_layout(xaxis_tickangle=-45)
    fig_geo.show()

In [12]:
# Analisi della distribuzione (Presenza vs Assenza di denunce)
# Trasformiamo numero_denunce in numero per il confronto
df_zero = con.execute(f"""
    SELECT 
        CASE 
            WHEN CAST(numero_denunce AS DOUBLE) = 0 THEN 'Zero Denunce' 
            ELSE 'Almeno 1 Denuncia' 
        END as status,
        COUNT(*) as conteggio
    FROM '{parquet_path}'
    GROUP BY status
""").df()

if df_zero.empty:
    print("⚠️ Nessun dato trovato per l'analisi della distribuzione.")
else:
    fig_pie = px.pie(df_zero, 
                    values='conteggio', 
                    names='status', 
                    title='Distribuzione delle segnalazioni (Presenza vs Assenza di reati)',
                    color_discrete_sequence=px.colors.qualitative.Pastel) # Colori più leggibili
    fig_pie.show()

In [14]:
# 1. Mapping aggiornato con i codici reali trovati nel tuo file
mapping_ita = {
    'ROBBER': 'Rapine',
    'MAFIAHOM': 'Omicidi di Mafia',
    'ATTEMPHOM': 'Tentati Omicidi',
    'CARTHEF': 'Furti d\'auto',
    'FOREARS': 'Incendi Boschivi', # Era ARSON
    'RAPE': 'Violenza Sessuale',
    'SHOPTHEF': 'Furti in esercizi commerciali',
    'VEHITHEF': 'Furti di veicoli',
    'KIDNAPP': 'Sequestro di Persona'
}

# 2. Query con CAST per evitare errori di tipo
# Nota: Usiamo '2015' per la PR, ma se testi in locale e hai solo il 2024, cambia temporaneamente l'anno
df_top_reati = con.execute(f"""
    SELECT 
        codice_reato, 
        SUM(CAST(numero_denunce AS DOUBLE)) as totale
    FROM '{parquet_path}'
    WHERE anno = '2015' 
      AND codice_reato NOT IN ('TOTAL', 'TOT') -- Escludiamo i totali
    GROUP BY codice_reato
    ORDER BY totale DESC
    LIMIT 10
""").df()

# 3. Applicazione del mapping
# Se il codice non è nel dizionario, mantiene il codice originale (es. 'MOPETHEF')
df_top_reati['reato_it'] = df_top_reati['codice_reato'].map(mapping_ita).fillna(df_top_reati['codice_reato'])

# 4. Grafico a ciambella
if df_top_reati.empty:
    print("⚠️ Attenzione: Nessun dato trovato per l'anno 2015. Se sei in locale, verifica se hai solo il 2024.")
else:
    fig_pie = px.pie(df_top_reati, 
                     values='totale', 
                     names='reato_it', 
                     title='Composizione dei principali reati denunciati (2015)',
                     hole=0.4, 
                     color_discrete_sequence=px.colors.sequential.RdBu,
                     template='plotly_white')
    
    fig_pie.update_traces(textposition='inside', textinfo='percent+label')
    fig_pie.show()

⚠️ Attenzione: Nessun dato trovato per l'anno 2015. Se sei in locale, verifica se hai solo il 2024.


In [16]:
# 1. Estrazione dati con doppia conversione (CAST)
# È fondamentale convertire numero_denunce in DOUBLE per poterlo sommare
df_gravi = con.execute(f"""
    SELECT 
        CAST(anno AS INTEGER) as anno, 
        codice_reato, 
        SUM(CAST(numero_denunce AS DOUBLE)) as totale
    FROM '{parquet_path}'
    WHERE codice_reato IN ('MAFIASS', 'INTENHOM', 'MAFIAHOM', 'KIDNAPP')
    GROUP BY anno, codice_reato
    ORDER BY anno
""").df()

# 2. AGGIUNGIAMO IL MAPPING
mapping_ita = {
    'INTENHOM': 'Omicidi volontari', 
    'MAFIASS': 'Associazione Mafiosa', 
    'MAFIAHOM': 'Omicidi di Mafia', 
    'KIDNAPP': 'Sequestro di Persona'
}

# Verifichiamo se abbiamo dati prima di mappare
if not df_gravi.empty:
    df_gravi['reato_it'] = df_gravi['codice_reato'].map(mapping_ita).fillna(df_gravi['codice_reato'])

    # 3. Creazione grafico
    fig_line = px.line(df_gravi, x='anno', y='totale', color='reato_it',
                      title='Evoluzione Reati di Grave Allarme Sociale (2010-2015)',
                      labels={'totale': 'Numero denunce', 'anno': 'Anno', 'reato_it': 'Tipo di reato'},
                      markers=True,
                      template='plotly_white')
    
    # Miglioriamo la visualizzazione dell'asse X
    fig_line.update_layout(xaxis_type='linear') 
    fig_line.show()
else:
    print("⚠️ Grafico vuoto: i codici specificati non sono stati trovati nell'anno selezionato.")
    print("Prova a controllare i codici presenti con: con.execute('SELECT DISTINCT codice_reato FROM ...').df()")